# FPN Weight Ablation Study
## FPN-LayerCAM & FPN-ScoreCAM — All 4 DenseBlocks × 3 Weight Configurations

**Model:** CheXpert-finetuned DenseNet169  
**Dataset:** Full CheXlocalize frontal test set (413 images)  
**Metrics:** All 5 — Pointing Game, IoU@0.5, mIoU, Deletion AUC, Insertion AUC  

---
### Ablation design
All configurations use **all 4 dense blocks** (denseblock1–4).  
Three weight sets are tested per method:

| Config | db1 | db2 | db3 | db4 | Rationale |
|---|---|---|---|---|---|
| Equal | 0.25 | 0.25 | 0.25 | 0.25 | Neutral baseline |
| Depth-proportional | 0.10 | 0.20 | 0.30 | 0.40 | Higher weight to deeper semantic layers |
| Reverse | 0.40 | 0.30 | 0.20 | 0.10 | Higher weight to low-level spatial detail |

**Total configurations:** 2 methods × 3 weight sets = **6 runs**  
Compared against the **original 3-level config** (db1=0.20, db2=0.35, db4=0.45)  
as a seventh reference point loaded from the expanded evaluation results.

## 0. Imports & Reproducibility

In [ ]:
import gc, json, random, warnings
from copy import deepcopy
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from PIL import Image
from sklearn.linear_model import Ridge
from sklearn.preprocessing import normalize as sk_normalise
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.models as models

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'  GPU  : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 1. Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
CHEXPERT_CKPT       = Path('./chexpert_output/best_model.pth')
GT_ANNOTATIONS_PATH = Path('./gt_annotations_test.json')

# CheXlocalize 413-image manifest + images (built in expanded eval notebook)
CHEX_413_DIR        = Path('./xai_evaluation_full/chexlocalize_images')

# Prior results from expanded evaluation (used for original-3level reference)
PRIOR_RESULTS_PATH  = Path('./xai_evaluation_full/results_chexlocalize_413.csv')

# Ablation output
ABLATION_DIR        = Path('./xai_ablation')
ABLATION_DIR.mkdir(parents=True, exist_ok=True)

# ── Labels ─────────────────────────────────────────────────────────────────────
LABELS: List[str] = [
    'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
    'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia',
    'Atelectasis', 'Pneumothorax', 'Effusion',
    'Pleural Other', 'Fracture', 'Support Devices', 'No Finding',
]
NUM_CLASSES = len(LABELS)

LABEL_MAP: Dict[str, str] = {
    'Enlarged Cardiomediastinum': 'Enlarged Cardiomediastinum',
    'Cardiomegaly'              : 'Cardiomegaly',
    'Lung Opacity'              : 'Airspace Opacity',
    'Lung Lesion'               : 'Lung Lesion',
    'Edema'                     : 'Edema',
    'Consolidation'             : 'Consolidation',
    'Atelectasis'               : 'Atelectasis',
    'Pneumothorax'              : 'Pneumothorax',
    'Effusion'                  : 'Pleural Effusion',
    'Support Devices'           : 'Support Devices',
}

# ── Ablation configurations ────────────────────────────────────────────────────
# All use all 4 dense blocks. Weights must sum to 1.0.
# Format: list of (layer_name, weight) tuples
ABLATION_CONFIGS: Dict[str, List[Tuple[str, float]]] = {
    'Equal': [
        ('features.denseblock1', 0.25),
        ('features.denseblock2', 0.25),
        ('features.denseblock3', 0.25),
        ('features.denseblock4', 0.25),
    ],
    'Depth-proportional': [
        ('features.denseblock1', 0.10),
        ('features.denseblock2', 0.20),
        ('features.denseblock3', 0.30),
        ('features.denseblock4', 0.40),
    ],
    'Reverse': [
        ('features.denseblock1', 0.40),
        ('features.denseblock2', 0.30),
        ('features.denseblock3', 0.20),
        ('features.denseblock4', 0.10),
    ],
}

# Validate weights sum to 1.0
for name, cfg in ABLATION_CONFIGS.items():
    total = sum(w for _,w in cfg)
    assert abs(total-1.0) < 1e-6, f'{name} weights sum to {total}, not 1.0'
    print(f'  {name:<22s}: {[f"{l.split(".")[-1]}={w}" for l,w in cfg]}')

# Methods to ablate
ABLATION_METHODS = ['FPN-LayerCAM', 'FPN-ScoreCAM']

# ── Hyperparameters ────────────────────────────────────────────────────────────
IMG_SIZE       = 224
THRESHOLD      = 0.5
SCORECAM_BATCH = 16
DEL_INS_STEPS  = 50
DEL_INS_BATCH  = 20
BLUR_KERNEL    = 51
IMAGENET_MEAN  = [0.485, 0.456, 0.406]
IMAGENET_STD   = [0.229, 0.224, 0.225]

print('\nConfiguration ready.')
print(f'  Ablation configs: {list(ABLATION_CONFIGS.keys())}')
print(f'  Ablation methods: {ABLATION_METHODS}')
print(f'  Total runs: {len(ABLATION_CONFIGS)*len(ABLATION_METHODS)}')

## 2. Architecture Verification

In [ ]:
def verify_architecture() -> Dict[str, dict]:
    """
    Programmatically verifies spatial dimensions and channel counts
    for all 4 dense blocks used in the ablation.
    """
    _m = models.densenet169(weights=None).eval()
    shapes: Dict[str, tuple] = {}
    hooks  = []
    for blk in ['denseblock1','denseblock2','denseblock3','denseblock4']:
        layer = getattr(_m.features, blk)
        h = layer.register_forward_hook(
            lambda m,i,o,b=blk: shapes.update({b: tuple(o.shape)})
        )
        hooks.append(h)
    with torch.no_grad():
        _m(torch.zeros(1,3,IMG_SIZE,IMG_SIZE))
    for h in hooks: h.remove()
    del _m

    nominal_strides = {
        'denseblock1':4,'denseblock2':8,'denseblock3':16,'denseblock4':32
    }
    info = {}
    print(f'DenseNet169 all-block verification (input {IMG_SIZE}×{IMG_SIZE}):')
    print(f'{"Block":<16} {"H×W":>8} {"Channels":>10} {"Stride":>8}')
    print('-'*46)
    for blk, shape in shapes.items():
        _,C,H,W = shape
        s = nominal_strides[blk]
        # Cross-check: input/H should equal stride
        actual_stride = IMG_SIZE // H
        match = '✓' if actual_stride==s else f'! actual={actual_stride}'
        print(f'{blk:<16} {H}×{W:>3}      {C:>10}  ×{s:>6}  {match}')
        info[blk] = {'channels':C,'spatial':(H,W),'stride':s}

    # Confirm all 4 blocks are present in each ablation config
    for cfg_name, cfg in ABLATION_CONFIGS.items():
        for layer_name, _ in cfg:
            blk = layer_name.split('.')[-1]
            assert blk in info, f'Config {cfg_name} references unknown block {blk}'
    print('\nAll ablation config layers verified against architecture.')
    return info


ARCH_INFO = verify_architecture()

## 3. Model Loading

In [ ]:
def load_model(ckpt_path: Path) -> nn.Module:
    model = models.densenet169(weights=None)
    in_feat = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.5),
        nn.Linear(in_feat, NUM_CLASSES),
    )
    ckpt  = torch.load(ckpt_path, map_location='cpu')
    model.load_state_dict(ckpt.get('state_dict', ckpt), strict=True)
    model.eval()
    print(f'Loaded checkpoint — epoch {ckpt.get("epoch","?")}, '
          f'val AUC {ckpt.get("val_auc","?")}')
    return model


model = load_model(CHEXPERT_CKPT).to(DEVICE)

## 4. Utilities (image, GT mask, metrics)

In [ ]:
PREPROCESS = T.Compose([
    T.Resize((IMG_SIZE,IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def load_image(path):
    pil = Image.open(path).convert('RGB').resize((IMG_SIZE,IMG_SIZE))
    return PREPROCESS(pil).unsqueeze(0).to(DEVICE), np.array(pil,dtype=np.uint8)

def normalise_cam(cam):
    mn,mx = cam.min(),cam.max()
    if mx-mn<1e-8: return np.zeros_like(cam,dtype=np.float32)
    return ((cam-mn)/(mx-mn)).astype(np.float32)

@torch.no_grad()
def get_probs(model,tensor):
    return model(tensor).sigmoid().squeeze(0).cpu().numpy()

# GT masks
with open(GT_ANNOTATIONS_PATH) as f:
    GT_ANNOTATIONS = json.load(f)

def rasterise(img_key, gt_label, out_size=IMG_SIZE):
    if img_key not in GT_ANNOTATIONS: return None
    entry = GT_ANNOTATIONS[img_key]
    if gt_label not in entry: return None
    H,W   = entry['img_size']
    mask  = np.zeros((H,W),dtype=np.uint8)
    for poly in entry[gt_label]:
        pts = np.array(poly,dtype=np.float32)
        if len(pts)>=3:
            cv2.fillPoly(mask,[pts.astype(np.int32).reshape(-1,1,2)],1)
    if mask.sum()==0: return None
    return cv2.resize(mask,(out_size,out_size),
                      interpolation=cv2.INTER_NEAREST).astype(np.uint8)

def filename_to_gt_key(fn): return Path(fn).stem

# Metrics
def pointing_game(cam, gt):
    for r,c in np.argwhere(cam==cam.max()):
        if gt[r,c]==1: return 1.0
    return 0.0

def iou_at_t(cam, gt, t=0.5):
    pred=(cam>=t).astype(np.uint8)
    inter=(pred&gt).sum(); union=(pred|gt).sum()
    return float(inter)/float(union) if union>0 else float('nan')

def mean_iou(cam, gt, ts=np.linspace(0.1,0.9,9)):
    vals=[iou_at_t(cam,gt,t) for t in ts]
    valid=[v for v in vals if not np.isnan(v)]
    return float(np.mean(valid)) if valid else float('nan')

def _blur_base(tensor,k=BLUR_KERNEL):
    k=k|1
    return F.avg_pool2d(tensor,k,stride=1,padding=k//2,
                        count_include_pad=False).detach()

@torch.no_grad()
def del_ins_auc(model,tensor,cam,class_idx,
                n=DEL_INS_STEPS,bs=DEL_INS_BATCH):
    H,W    = cam.shape
    order  = np.argsort(cam.ravel())[::-1]
    steps  = np.linspace(0,H*W,n+1,dtype=int)
    mean_v = tensor.mean(dim=(0,2,3),keepdim=True).expand_as(tensor).clone()
    blur   = _blur_base(tensor)
    d_s, i_s, bd, bi = [], [], [], []
    for k_px in steps:
        d=tensor.clone(); iv=blur.clone()
        if k_px>0:
            r=order[:k_px]//W; c=order[:k_px]%W
            d[0,:,r,c]=mean_v[0,:,r,c]; iv[0,:,r,c]=tensor[0,:,r,c]
        bd.append(d); bi.append(iv)
        if len(bd)==bs or k_px==steps[-1]:
            ds =model(torch.cat(bd)).sigmoid()[:,class_idx].cpu().numpy()
            is_=model(torch.cat(bi)).sigmoid()[:,class_idx].cpu().numpy()
            d_s.extend(ds.tolist()); i_s.extend(is_.tolist())
            bd, bi = [], []
    xs = np.linspace(0,1,len(d_s))
    return float(np.trapz(d_s,xs)), float(np.trapz(i_s,xs))

print('Utilities ready.')

## 5. Ablation CAM Functions

In [ ]:
class ActivationGradientHook:
    def __init__(self, layer):
        self.activation = self.gradient = None
        self._fwd = layer.register_forward_hook(
            lambda m,i,o: setattr(self,'activation',o.detach()))
        self._bwd = layer.register_full_backward_hook(
            lambda m,gi,go: setattr(self,'gradient',go[0].detach()))
    def remove(self): self._fwd.remove(); self._bwd.remove()

def get_layer(model,name):
    m=model
    for p in name.split('.'): m=getattr(m,p)
    return m


def fpn_layercam_config(
    model, tensor, class_idx,
    fpn_config: List[Tuple[str,float]],
) -> np.ndarray:
    """
    FPN-LayerCAM for an arbitrary list of (layer_name, weight) pairs.
    Separate backward pass per level ensures decoupled gradient estimates
    (critical for DenseNet's skip connections — see notebook 1 for rationale).
    """
    fused = np.zeros((IMG_SIZE,IMG_SIZE), dtype=np.float32)
    for layer_name, weight in fpn_config:
        hook = ActivationGradientHook(get_layer(model,layer_name))
        model.zero_grad()
        model(tensor)[0,class_idx].backward(retain_graph=True)
        act  = hook.activation.squeeze(0)
        grad = hook.gradient.squeeze(0)
        hook.remove()
        cam  = F.relu((act*F.relu(grad)).sum(0)).cpu().numpy()
        cam  = cv2.resize(cam,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR)
        fused += weight * normalise_cam(cam)
    model.zero_grad()
    return normalise_cam(fused)


def scorecam_single(model,tensor,class_idx,layer_name,batch_size=SCORECAM_BATCH):
    """ScoreCAM at a single layer."""
    acts={}
    h=get_layer(model,layer_name).register_forward_hook(
        lambda m,i,o: acts.update({'v':o.detach()}))
    with torch.no_grad(): model(tensor)
    h.remove()
    activations=acts['v'].squeeze(0)
    n_ch=activations.shape[0]
    with torch.no_grad():
        baseline=model(torch.zeros_like(tensor)).sigmoid()[0,class_idx].item()
    ch_w=np.zeros(n_ch,dtype=np.float32)
    for bs in range(0,n_ch,batch_size):
        be=min(bs+batch_size,n_ch)
        masked=[]
        for c in range(bs,be):
            ch=activations[c].cpu().numpy()
            ch=cv2.resize(ch,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR)
            mn,mx=ch.min(),ch.max()
            ch=(ch-mn)/(mx-mn+1e-8)
            masked.append(tensor*torch.from_numpy(ch).float().to(DEVICE)[None,None])
        with torch.no_grad():
            s=model(torch.cat(masked)).sigmoid()[:,class_idx].cpu().numpy()
        ch_w[bs:be]=s-baseline
    wt =torch.from_numpy(ch_w).to(activations.device)
    cam=F.relu((F.relu(wt)[:,None,None]*activations).sum(0)).cpu().numpy()
    return normalise_cam(cv2.resize(cam,(IMG_SIZE,IMG_SIZE),
                                    interpolation=cv2.INTER_LINEAR))


def fpn_scorecam_config(
    model, tensor, class_idx,
    fpn_config: List[Tuple[str,float]],
) -> np.ndarray:
    """
    FPN-ScoreCAM for an arbitrary (layer_name, weight) config.
    Each level runs independently; maps are normalised before fusion.
    """
    fused = np.zeros((IMG_SIZE,IMG_SIZE), dtype=np.float32)
    for layer_name, weight in fpn_config:
        cam  = scorecam_single(model,tensor,class_idx,layer_name)
        fused += weight * cam
    return normalise_cam(fused)


def compute_ablation_cam(
    method      : str,
    fpn_config  : List[Tuple[str,float]],
    model       : nn.Module,
    tensor      : torch.Tensor,
    class_idx   : int,
) -> np.ndarray:
    """Dispatches to the correct CAM function for the given method."""
    if method == 'FPN-LayerCAM':
        return fpn_layercam_config(model,tensor,class_idx,fpn_config)
    elif method == 'FPN-ScoreCAM':
        return fpn_scorecam_config(model,tensor,class_idx,fpn_config)
    else:
        raise ValueError(f'Unknown ablation method: {method}')

print('Ablation CAM functions ready.')

## 6. Per-Image Ablation Evaluation

In [ ]:
def evaluate_image_ablation(
    model     : nn.Module,
    img_path  : Path,
    img_key   : str,
    method    : str,
    config_name: str,
    fpn_config: List[Tuple[str,float]],
) -> List[dict]:
    """
    Evaluates one (method, config) combination on a single image.
    Returns a list of result dicts (one per predicted class).
    """
    tensor, _ = load_image(img_path)
    probs     = get_probs(model, tensor)
    pred_ids  = [i for i,p in enumerate(probs) if p>=THRESHOLD] or [int(np.argmax(probs))]

    rows = []
    for class_idx in pred_ids:
        label = LABELS[class_idx]

        gt_mask = None
        if label in LABEL_MAP:
            gt_mask = rasterise(img_key, LABEL_MAP[label])

        cam = compute_ablation_cam(method, fpn_config, model, tensor, class_idx)

        pg   = pointing_game(cam,gt_mask) if gt_mask is not None else float('nan')
        iou  = iou_at_t(cam,gt_mask)      if gt_mask is not None else float('nan')
        miou = mean_iou(cam,gt_mask)       if gt_mask is not None else float('nan')
        d, i = del_ins_auc(model,tensor,cam,class_idx)

        rows.append({
            'filename'    : img_path.name,
            'img_key'     : img_key,
            'method'      : method,
            'config'      : config_name,
            'label'       : label,
            'class_idx'   : class_idx,
            'prob'        : round(float(probs[class_idx]),4),
            'gt_available': gt_mask is not None,
            'pointing_game': round(pg,4)   if not np.isnan(pg)   else float('nan'),
            'iou_05'      : round(iou,4)   if not np.isnan(iou)  else float('nan'),
            'miou'        : round(miou,4)  if not np.isnan(miou) else float('nan'),
            'deletion_auc': round(d,4),
            'insertion_auc': round(i,4),
        })
    torch.cuda.empty_cache(); gc.collect()
    return rows

## 7. Ablation Batch Runner

In [ ]:
def run_ablation_config(
    model       : nn.Module,
    manifest_path: Path,
    image_dir   : Path,
    method      : str,
    config_name : str,
    fpn_config  : List[Tuple[str,float]],
    results_path: Path,
) -> pd.DataFrame:
    """
    Runs ablation evaluation for one (method, config) pair on all images
    in manifest_path. Resume-friendly: skips already-evaluated images.
    Results are appended incrementally to results_path.
    """
    manifest = pd.read_csv(manifest_path)

    if results_path.exists():
        existing   = pd.read_csv(results_path)
        # Resume: check which (filename, method, config) combos are done
        done = set(
            zip(existing['filename'], existing['method'], existing['config'])
        )
        print(f'  Resuming — {len(done)} (file,method,config) rows already done.')
    else:
        done = set()
        pd.DataFrame(columns=[
            'filename','img_key','method','config','label','class_idx','prob',
            'gt_available','pointing_game','iou_05','miou',
            'deletion_auc','insertion_auc'
        ]).to_csv(results_path, index=False)

    skipped = []
    for _, row in tqdm(manifest.iterrows(), total=len(manifest),
                       desc=f'{method} / {config_name}'):
        fname = row['filename']
        if (fname, method, config_name) in done:
            continue
        img_path = image_dir / fname
        if not img_path.exists():
            skipped.append(fname); continue

        img_key = filename_to_gt_key(fname)
        try:
            result_rows = evaluate_image_ablation(
                model, img_path, img_key,
                method, config_name, fpn_config
            )
            pd.DataFrame(result_rows).to_csv(
                results_path, mode='a', header=False, index=False)
        except Exception as e:
            print(f'    ERROR {fname}: {e}')
            skipped.append(fname)

    if skipped:
        print(f'  Skipped {len(skipped)} images.')
    return pd.read_csv(results_path)

## 8. Run All Ablation Configurations

In [ ]:
ABLATION_RESULTS_PATH = ABLATION_DIR / 'ablation_results.csv'
MANIFEST_PATH         = CHEX_413_DIR / 'manifest.csv'

# Run all 6 combinations: 2 methods × 3 configs
for method in ABLATION_METHODS:
    for config_name, fpn_config in ABLATION_CONFIGS.items():
        print(f'\n=== {method} / {config_name} ===')
        print(f'    Layers: {[(l.split(".")[-1],w) for l,w in fpn_config]}')
        ablation_df = run_ablation_config(
            model        = model,
            manifest_path= MANIFEST_PATH,
            image_dir    = CHEX_413_DIR,
            method       = method,
            config_name  = config_name,
            fpn_config   = fpn_config,
            results_path = ABLATION_RESULTS_PATH,
        )
        # Quick summary after each config
        sub  = ablation_df[
            (ablation_df['method']==method) &
            (ablation_df['config']==config_name)
        ]
        mask = sub[sub['gt_available']==True]
        print(f'    PG={mask["pointing_game"].mean():.3f}  '
              f'mIoU={mask["miou"].mean():.3f}  '
              f'Del={sub["deletion_auc"].mean():.3f}  '
              f'Ins={sub["insertion_auc"].mean():.3f}')

print('\nAll ablation runs complete.')
ablation_df = pd.read_csv(ABLATION_RESULTS_PATH)
print(f'Total result rows: {len(ablation_df)}')

## 9. Load Original 3-Level Reference Results

In [ ]:
# Load FPN-LayerCAM and FPN-ScoreCAM rows from the expanded evaluation
# These are the original 3-level config (db1=0.20, db2=0.35, db4=0.45)
if PRIOR_RESULTS_PATH.exists():
    prior_df = pd.read_csv(PRIOR_RESULTS_PATH)
    ref_rows = prior_df[prior_df['method'].isin(ABLATION_METHODS)].copy()
    ref_rows['config'] = 'Original-3level'
    print(f'Loaded {len(ref_rows)} reference rows from expanded evaluation.')
else:
    print('WARNING: Prior results not found. Run expanded evaluation notebook first.')
    ref_rows = pd.DataFrame()

# Merge ablation + reference into one dataframe
all_results = pd.concat([ablation_df, ref_rows], ignore_index=True)

# Config display order
CONFIG_ORDER = ['Original-3level','Equal','Depth-proportional','Reverse']

print(f'Combined results: {len(all_results)} rows')
print(f'Configs present: {all_results["config"].unique().tolist()}')

## 10. Ablation Summary Tables

In [ ]:
METRICS_ALL = ['pointing_game','iou_05','miou','deletion_auc','insertion_auc']


def ablation_summary_table(
    df     : pd.DataFrame,
    method : str,
    configs: List[str] = CONFIG_ORDER,
) -> pd.DataFrame:
    """
    Rows = configs, Cols = metrics.
    Values = mean ± std.
    Mask-based metrics computed only on gt_available=True rows.
    """
    sub = df[df['method']==method]
    rows = []
    for cfg in configs:
        cfg_df = sub[sub['config']==cfg]
        if len(cfg_df)==0: continue
        mask_df = cfg_df[cfg_df['gt_available']==True]
        row = {'Config': cfg}
        for met in METRICS_ALL:
            src = mask_df if met in ['pointing_game','iou_05','miou'] else cfg_df
            v   = src[met].dropna()
            row[met] = f'{v.mean():.4f}±{v.std():.4f}' if len(v) else 'N/A'
        rows.append(row)
    tbl = pd.DataFrame(rows).set_index('Config')
    print(f'\n{method} — ablation summary')
    display(tbl.style
              .set_caption(f'{method} weight ablation (CheXlocalize 413)')
           )
    return tbl


tbl_lc = ablation_summary_table(all_results, 'FPN-LayerCAM')
tbl_sc = ablation_summary_table(all_results, 'FPN-ScoreCAM')

tbl_lc.to_csv(ABLATION_DIR / 'ablation_summary_FPNLayerCAM.csv')
tbl_sc.to_csv(ABLATION_DIR / 'ablation_summary_FPNScoreCAM.csv')
print('\nSummary tables saved.')

## 11. Per-Label Ablation Tables

In [ ]:
def per_label_ablation_table(
    df:     pd.DataFrame,
    method: str,
    metric: str,
    configs: List[str] = CONFIG_ORDER,
) -> pd.DataFrame:
    """
    Rows = labels, Cols = configs.
    Values = mean metric per (label, config).
    """
    sub = df[df['method']==method]
    if metric in ['pointing_game','iou_05','miou']:
        sub = sub[sub['gt_available']==True]
    tbl = sub.groupby(['label','config'])[metric].mean().unstack('config')
    # Reorder columns to CONFIG_ORDER (keep only those present)
    tbl = tbl[[c for c in configs if c in tbl.columns]]
    return tbl.round(4)


for method in ABLATION_METHODS:
    for metric in METRICS_ALL:
        tbl = per_label_ablation_table(all_results, method, metric)
        safe_method = method.replace('-','_').replace(' ','_')
        tbl.to_csv(ABLATION_DIR / f'per_label_{safe_method}_{metric}.csv')

print('Per-label tables saved.')

# Display mIoU per label for FPN-LayerCAM (most informative)
tbl_miou_lc = per_label_ablation_table(all_results,'FPN-LayerCAM','miou')
print('\nFPN-LayerCAM mIoU per label × config:')
display(tbl_miou_lc.style
          .highlight_max(axis=1, color='#c6efce')
          .highlight_min(axis=1, color='#ffeb9c')
          .format('{:.4f}',na_rep='N/A')
          .set_caption('FPN-LayerCAM mIoU — rows=labels, cols=weight configs')
       )

tbl_miou_sc = per_label_ablation_table(all_results,'FPN-ScoreCAM','miou')
print('\nFPN-ScoreCAM mIoU per label × config:')
display(tbl_miou_sc.style
          .highlight_max(axis=1, color='#c6efce')
          .highlight_min(axis=1, color='#ffeb9c')
          .format('{:.4f}',na_rep='N/A')
          .set_caption('FPN-ScoreCAM mIoU — rows=labels, cols=weight configs')
       )

## 12. Visualisations

In [ ]:
CONFIG_COLORS = {
    'Original-3level'   : '#378ADD',
    'Equal'             : '#1D9E75',
    'Depth-proportional': '#D85A30',
    'Reverse'           : '#7F77DD',
}


def plot_ablation_bars(df, method, filepath):
    """
    5-panel bar chart: one panel per metric.
    Bars = configs, coloured by CONFIG_COLORS.
    """
    present_configs = [c for c in CONFIG_ORDER if c in df['config'].unique()]
    sub = df[df['method']==method]

    metric_info = [
        ('pointing_game', 'Pointing Game ↑', True),
        ('iou_05',        'IoU@0.5 ↑',       True),
        ('miou',          'mIoU ↑',           True),
        ('deletion_auc',  'Deletion AUC ↓',   False),
        ('insertion_auc', 'Insertion AUC ↑',  False),
    ]
    fig, axes = plt.subplots(1,5,figsize=(22,5))

    for ax,(met,title,use_mask) in zip(axes,metric_info):
        means, errs, cols = [], [], []
        for cfg in present_configs:
            cfg_df = sub[sub['config']==cfg]
            if use_mask: cfg_df = cfg_df[cfg_df['gt_available']==True]
            v = cfg_df[met].dropna()
            means.append(v.mean() if len(v) else 0)
            errs.append(v.std()   if len(v) else 0)
            cols.append(CONFIG_COLORS.get(cfg,'#888888'))

        bars = ax.bar(present_configs, means, color=cols,
                      yerr=errs, capsize=4,
                      edgecolor='black', linewidth=0.5)
        ax.set_title(title, fontsize=10)
        ax.tick_params(axis='x',rotation=30,labelsize=8)
        ax.grid(axis='y',alpha=0.3)
        for bar,v,e in zip(bars,means,errs):
            ax.text(bar.get_x()+bar.get_width()/2,
                    bar.get_height()+max(errs)*0.05+0.001,
                    f'{v:.3f}',ha='center',va='bottom',fontsize=8)

    plt.suptitle(f'{method} — Weight Ablation (CheXlocalize 413)',fontsize=12)
    plt.tight_layout()
    plt.savefig(filepath,dpi=150,bbox_inches='tight')
    plt.show()
    print(f'Saved → {filepath}')


plot_ablation_bars(all_results,'FPN-LayerCAM',
                  ABLATION_DIR/'ablation_bars_FPNLayerCAM.png')
plot_ablation_bars(all_results,'FPN-ScoreCAM',
                  ABLATION_DIR/'ablation_bars_FPNScoreCAM.png')


# ── Heatmaps: per-label × config ──────────────────────────────────────────────
for method in ABLATION_METHODS:
    for metric, cmap in [('miou','YlGn'),('pointing_game','YlGn'),
                          ('deletion_auc','YlOrRd_r'),('insertion_auc','YlGn')]:
        tbl = per_label_ablation_table(all_results, method, metric)
        if tbl.empty: continue
        fig,ax=plt.subplots(figsize=(len(tbl.columns)*1.8+2,len(tbl)*0.55+1.8))
        sns.heatmap(tbl.astype(float),annot=True,fmt='.3f',cmap=cmap,
                    linewidths=0.4,ax=ax,cbar_kws={'shrink':0.7},
                    annot_kws={'size':9})
        ax.set_title(f'{method} — {metric} per label × weight config',
                     fontsize=11,pad=10)
        ax.set_xlabel('Weight config'); ax.set_ylabel('Label')
        plt.tight_layout()
        safe_m = method.replace('-','_')
        out = ABLATION_DIR/f'heatmap_{safe_m}_{metric}.png'
        plt.savefig(out,dpi=150,bbox_inches='tight')
        plt.show()
        print(f'Saved → {out}')

## 13. Cross-Method Comparison: Original-3Level vs Best Ablation Config

In [ ]:
def best_config_per_metric(
    df:     pd.DataFrame,
    method: str,
) -> pd.DataFrame:
    """
    For each metric, identifies which weight config achieves the best score.
    Deletion AUC: lower is better. All others: higher is better.
    """
    sub      = df[df['method']==method]
    configs  = [c for c in CONFIG_ORDER if c in sub['config'].unique()]
    lower_is_better = {'deletion_auc'}
    rows = []
    for met in METRICS_ALL:
        use_mask = met in ['pointing_game','iou_05','miou']
        means = {}
        for cfg in configs:
            cfg_df = sub[sub['config']==cfg]
            if use_mask: cfg_df = cfg_df[cfg_df['gt_available']==True]
            v = cfg_df[met].dropna()
            means[cfg] = v.mean() if len(v) else float('nan')

        valid = {k:v for k,v in means.items() if not np.isnan(v)}
        if not valid: continue
        best_cfg = min(valid,key=valid.get) if met in lower_is_better \
                   else max(valid,key=valid.get)
        ref_val  = means.get('Original-3level', float('nan'))
        best_val = valid[best_cfg]
        delta    = best_val - ref_val if not np.isnan(ref_val) else float('nan')

        rows.append({
            'Metric'          : met,
            'Best config'     : best_cfg,
            'Best value'      : round(best_val,4),
            'Original-3level' : round(ref_val,4) if not np.isnan(ref_val) else 'N/A',
            'Delta vs original': f'{delta:+.4f}' if not np.isnan(delta) else 'N/A',
        })
    tbl = pd.DataFrame(rows).set_index('Metric')
    print(f'\n{method} — best config per metric vs original-3level')
    display(tbl)
    return tbl


best_lc = best_config_per_metric(all_results,'FPN-LayerCAM')
best_sc = best_config_per_metric(all_results,'FPN-ScoreCAM')

best_lc.to_csv(ABLATION_DIR/'best_config_FPNLayerCAM.csv')
best_sc.to_csv(ABLATION_DIR/'best_config_FPNScoreCAM.csv')
print('Best-config tables saved.')

## 14. Final Summary

In [ ]:
print('='*70)
print('  FPN Weight Ablation — Final Summary (CheXlocalize 413 images)')
print('='*70)
for method in ABLATION_METHODS:
    print(f'\n  {method}')
    print(f'  {"Config":<22}  {"PG":>6}  {"IoU@0.5":>8}  {"mIoU":>6}  '
          f'{"Del AUC":>8}  {"Ins AUC":>8}')
    print('  ' + '-'*64)
    sub = all_results[all_results['method']==method]
    for cfg in CONFIG_ORDER:
        cfg_df = sub[sub['config']==cfg]
        if len(cfg_df)==0: continue
        mask_df = cfg_df[cfg_df['gt_available']==True]
        pg   = mask_df['pointing_game'].mean()
        iou  = mask_df['iou_05'].mean()
        miou = mask_df['miou'].mean()
        d    = cfg_df['deletion_auc'].mean()
        i    = cfg_df['insertion_auc'].mean()
        print(f'  {cfg:<22}  {pg:>6.3f}  {iou:>8.3f}  {miou:>6.3f}  {d:>8.3f}  {i:>8.3f}')

print(f'\n  Outputs saved to: {ABLATION_DIR.resolve()}')
print('='*70)